# Creating the Alice Initial Commitment Transaction V3

In this section, we'll build a lightning channel initial commitment transaction v3 from scratch using Python. We'll walk through each part of the transaction — how it's constructed and signed. The process will be tested using Bitcoin Core in regtest mode.

## Setup

For this notebook, we’ll use the basepoints derivated in the `chapter 0 - lightning node keys derivation`, and the funding transaction created in the `chapter 1 - channel funding transaction`.

In [1]:
%run "../chapter 1 - channel funding transaction/funding transaction.ipynb"

2026-04-06T00:56:12.400000Z TestFramework (INFO): PRNG seed is: 4167363511980581611
2026-04-06T00:56:12.403000Z TestFramework (INFO): Initializing test directory /tmp/bitcoin_func_test_4kx7wtmi
🟢 New TestShell started. Block height: 0
Alice Per Commitment Seed 34b581ec20bf2c6cae3d4d4dcbfddc8a3727a1e9a57c55f3520e770607898c06
Bob Per Commitment Seed 89c994b3ddad4698acee71e42d8bcace48eea739caaba371eb110e77663ec56d
Alice Payment BasePoint:  025f892a06124391e2f38ce35d943cdc09f63e203330dbd9cb6113a903e0738458
Bob Payment BasePoint:  02f98efd3f2b2fbe7bd83c419f5f64f8280798b8a9175fdb77c0091bbb95c79506
To obscure commitment number 0xb433fd43a66f
Alice funding pubkey: ace9819d085bb560069919025cec1640bcc6a2cc0f0cfd342e78d7c6e25665f2
Alice funding privkey: bdce736bded8ab74665446a53b419283b6757068f79d2fbc60f4947832055ab5
Alice funding address: bcrt1p4n5cr8ggtw6kqp5eryp9emqkgz7vdgkvpux06dpw0rtudcjkvheqdwvfep
Alice sweeper pubkey: df7a8859536d3e673c9ff1e2ffd8ee4bda5795384416c015c1200ae98e364630
Alice s

In [2]:
from functions.test_framework.script import Tapbranch, TapTree, TapLeaf, CScript, TaprootSignatureHash, OP_CHECKSIG, OP_CHECKSIGVERIFY, OP_CHECKSEQUENCEVERIFY, OP_DROP, OP_16, OP_1, OP_SIZE, OP_EQUALVERIFY, OP_HASH160, OP_CHECKLOCKTIMEVERIFY
from functions.test_framework.messages import CTxInWitness, ser_string
TAPSCRIPT_VER = bytes([0xc0])

At this point, no HTLCs have been added yet, which makes the initial commitment transaction simpler. Only after Alice has a fully signed initial commitment transaction she will broadcast the funding transaction, this serves as a guarantee for Alice (the funder) that she can reclaim her sats back if anything goes wrong.

### The Commitment Unsigned Transaction

### The Input

The input is the channel funding transaction.

In [3]:
# VERSION
# version '3' indicates that we are using V3 TRUC (Topologically Restricted Until Confirmation) transactions
version = bytes.fromhex("0300 0000")

# MARKER (new to segwit)
marker = bytes.fromhex("00")

# FLAG (new to segwit)
flag = bytes.fromhex("01")

# INPUTS
# We have just 1 input
input_count = bytes.fromhex("01")

# Convert txid and index to bytes (little endian)
txid = (bytes.fromhex(funding_channel_txid))[::-1]
funding_channel_index = 0
index = funding_channel_index.to_bytes(4, byteorder="little", signed=False)

# For the unsigned transaction we use an empty scriptSig
scriptsig = bytes.fromhex("")

# sequence: upper 8 bits are 0x80, lower 24 bits are the upper 24 bits of the obscured commitment number
# Commitment number on the opening channel 
commitment_number = 0
# obscured commitment number is result of xor operation 
commitment_number_obscured = lower48_to_obscure ^ commitment_number
# Extract the upper 24 bits of the obscured commitment number
upper_24_bits = (commitment_number_obscured >> 24) & 0xFFFFFF
# Combine the upper 8 bits (0x80) with the lower 24 bits (upper 24 of obscured number)
sequence = (0x80 << 24) | upper_24_bits
# Convert to bytes (byte, big-endian)
sequence = sequence.to_bytes(4, byteorder='big')
# Convert to little-endian 
sequence = sequence[::-1]

inputs = (
    txid
    + index
    + varint_len(scriptsig)
    + scriptsig
    + sequence
)

print("Commitment Number Osbscured", hex(commitment_number_obscured))

Commitment Number Osbscured 0xb433fd43a66f


### The Outputs

The Basis of Lightning Technology ([BOLT 3]https://github.com/lightning/bolts/pull/1228/changes#diff-6bed824879b760d329ec379b16a1d0e78ffba034fdfa13b95cf0480e09fa7c4b) defines the outputs as following:

* if the amount of the commitment transaction to_local output would be less than dust_limit_satoshis set by the transaction owner:
    * MUST NOT contain that output.
* otherwise:
    * MUST be generated as specified in to_local Output.

* if the amount of the commitment transaction to_remote output would be less than dust_limit_satoshis set by the transaction owner:
    * MUST NOT contain that output.
* otherwise:
    * MUST be generated as specified in to_remote Output.

* for every offered HTLC:
    * if the HTLC amount minus the HTLC-timeout fee would be less than dust_limit_satoshis set by the transaction owner:
        * MUST NOT contain that output.
    * otherwise:
        * MUST be generated as specified in Offered HTLC Outputs.

* for every received HTLC:
    * if the HTLC amount minus the HTLC-success fee would be less than dust_limit_satoshis set by the transaction owner:
        * MUST NOT contain that output.
    * otherwise:
        * MUST be generated as specified in Received HTLC Outputs.
        
* if zero_fee_commitments applies:
    * MUST add a [shared_anchor](https://github.com/t-bast/bolts/blob/b982ae78035cd8f51ed5f9bd64593d11600b390e/03-transactions.md#shared_anchor-output-zero_fee_commitments) output with an amount computed as detailed in the shared_anchor section.

Since this is Alice’s first commitment transaction, it will have no outputs to Bob. That’s because Alice is the funder of the channel and isn't sending any sats to Bob initially. If she wanted to transfer funds at channel opening, she could have set the `push_msat` field in the `open_channel` message, specifying the amount in millisatoshis.

Additionally, because the channel hasn’t been opened yet, there are no offered or received HTLCs — so no HTLC outputs will be created either.

As a result, the outputs in the first commitment transaction will be as follows:

Alice first Commitment Transaction will have two outputs:
* to_local_anchor_output
* shared_ancor_output

The Basis of Lightning Technology ([BOLT 3](https://github.com/lightning/bolts/blob/master/03-transactions.md)) specifies that transaction outputs must be sorted by value, from smallest to largest.

A **trimmed HTLC** is an HTLC that does not get included in the commitment transaction because its value is too low to be economically spent. Specifically, if the value of the HTLC is below the dust limit plus the estimated fee required to claim it, the output is trimmed — in other words, left out of the transaction entirely.

This mechanism helps avoid bloating the commitment transaction with outputs that would either be unspendable or cost more in fees than they're worth.


### Alice First Commitment Transaction Outputs

#### Shared Anchor Output

This is the output that anyone will be able to use to CPFP the commitment transaction.

    +------+---------------+
    | OP_1 |    <0x4e73>   |
    +------+---------------+

#### To Local Output

The to_local output is responsible for paying the local peer their channel balance. The output must be revocable by the remote party at all times and only after to_self_delay blocks should the local party be able to spend from the output.

As you can see in the diagram below, both these paths are added as Taproot leaves in the Taproot tree and a public [NUMS](https://github.com/lightninglabs/lightning-node-connect/tree/master/mailbox/numsgen) (Nothing Up My Sleeve) point is used as the internal key which effectively cancels out the key-spend path.


    +------+---------------+
    | OP_1 |       Q       |
    +------+---------------+
                   ^
                   |   +-------------------+
                    ---| P(revocation) + T |
                       +-------------------+
                                ^
                                |
                          +-----------+        
                          | T = t * G |
                          +-----------+        
                                ^
                                |
     +---+   +----------------------------------------------+
     | t | = | TaggedHash ("Taptweak", NUMS || script_root) |
     +---+   +----------------------------------------------+
                                                   ^
                                                   |
                                                   |
                                                   |
                                                   |
                                                +----+
                                                | hA |
                                                +----+
                                                   ^
                                                   |
                                 +------------------------------+
                                 | P(local_delayed) OP_CHECKSIG |
                                 |                              |
                                 | to_self_delay OP_CSV OP_DROP |
                                 +------------------------------+

#### Key Derivations

Each commitment transaction uses unique keys: `localpubkey`, `local_htlcpubkey`, `remote_htlcpubkey`, `local_delayedpubkey`, and `remote_delayedpubkey`. These public keys are derived by simply adding a per-commitment point to their respective base points. As defined at Basis of Lightning Technology ([BOLT 3](https://github.com/lightning/bolts/blob/master/03-transactions.md#key-derivation)):

```
pubkey = basepoint + SHA256(per_commitment_point || basepoint) * G

```

- The `localpubkey` uses the local node's `payment_basepoint`;
- The `local_htlcpubkey` uses the local node's `htlc_basepoint`;
- The `remote_htlcpubkey` uses the remote node's `htlc_basepoint`;
- The `local_delayedpubkey` uses the local node's `delayed_payment_basepoint`;
- The `remote_delayedpubkey` uses the remote node's `delayed_payment_basepoint`.

The `revocationpubkey` is a blinded public key. When the local node prepares a new commitment transaction for the remote node, it derives the `revocationpubkey` by combining its own `revocation_basepoint` with the remote node’s `per_commitment_point`.

```
revocationpubkey = revocation_basepoint * SHA256(revocation_basepoint || per_commitment_point) + per_commitment_point * SHA256(per_commitment_point || revocation_basepoint)
```

Later, when the remote node revokes that commitment by revealing the corresponding `per_commitment_secret`, the local node can derive the `revocationprivkey`. At that point, it possesses both secrets needed for the derivation: the `revocation_basepoint_secret` and the `per_commitment_secret`.

This construction ensures that neither the node providing the `basepoint` nor the node providing the `per_commitment_point` can derive the private key on their own—each requires the other’s secret.

```
revocationprivkey = revocation_basepoint_secret * SHA256(revocation_basepoint || per_commitment_point) + per_commitment_secret * SHA256(per_commitment_point || revocation_basepoint)
```

In [4]:
# Outputs for Alice First Commitment Transaction
# 0x02 outputs
output_count = bytes.fromhex("02")

# SHARED ANCHOR OUTPUT
# Shared Anchor output amount
anchor_output_value_satoshis = 240
anchor_output_value = anchor_output_value_satoshis.to_bytes(8, byteorder="little", signed=False)

# <0x4e73>
script = CScript([b'\x4e\x73'])

# scriptPubKey P2A: OP_1 (0x51) + PUSH02 (0x02) + (0x4e) + (0x73)
shared_anchor_spk = bytes.fromhex("51") + script
print("Shared Anchor ScriptPubKey:", shared_anchor_spk.hex())

Shared Anchor ScriptPubKey: 51024e73


In [7]:
# Create Alice per-commitment
alice_per_commitment = per_commitment(alice_per_commitment_seed, commitment_number)
# Create Bob per-commitment
bob_per_commitment = per_commitment(bob_per_commitment_seed, commitment_number)

# Create Alice Delayed Public Key
alice_delayed_pubkey = derivate_key(alice_node_seed, family=4, channel_index=0).get_pubkey(alice_per_commitment.get_pub())
print(f"Alice Delayed PubKey: {alice_delayed_pubkey.get_bytes(bip340=True).hex()}")

# Create Bob Revocation Public Key
bob_revocation_pubkey = derivate_revocation_key(bob_node_seed, channel_index=0).get_pubkey(alice_per_commitment.get_pub())
print(f"Bob Revocation PubKey: {bob_revocation_pubkey.get_bytes(bip340=True).hex()}")


# TO LOCAL OUTPUT
# to local output amount
to_local_value_sat = channel_value_sat - anchor_output_value_satoshis
to_local_value = to_local_value_sat.to_bytes(8, byteorder="little", signed=False)

# Create the leaf script
# P(local_delayed) OP_CHECKSIG to_self_delay OP_CSV OP_DROP
to_self_delay = 144
script = CScript([alice_delayed_pubkey.get_bytes(bip340=True), OP_CHECKSIG, to_self_delay, OP_CHECKSEQUENCEVERIFY, OP_DROP])

# Compute TapLeave
# Method: ser_string(data) is a function which adds compactsize to input data.
hash_input = TAPSCRIPT_VER + ser_string(script)
taggedhash_leaf = tagged_hash("TapLeaf", hash_input)

# Compute TapTweak
taptweak = tagged_hash("TapTweak", bob_revocation_pubkey.get_bytes(bip340=True) + taggedhash_leaf)
revocation_tweaked = bob_revocation_pubkey.tweak_add(taptweak)

# scriptPubKey P2TR: OP_1 (0x51) + PUSH32 (0x20) + xonly(32B)
alice_to_local_spk = bytes.fromhex("51") + varint_len(revocation_tweaked.get_bytes(bip340=True)) + revocation_tweaked.get_bytes(bip340=True)

outputs = (
    anchor_output_value
    + varint_len(shared_anchor_spk)
    + shared_anchor_spk
    + to_local_value
    + varint_len(alice_to_local_spk)
    + alice_to_local_spk
)

# Locktime: upper 8 bits are 0x20, lower 24 bits are the lower 24 bits of the obscured commitment number
# Extract the lower 24 bits of the obscured commitment number
lower_24_bits = commitment_number_obscured & 0xFFFFFF
# Combine the upper 8 bits (0x20) with the lower 24 bits (lower 24 of obscured number)
locktime = (0x20 << 24) | lower_24_bits
# Convert to bytes (1 byte, big-endian)
locktime = locktime.to_bytes(4, byteorder='big')
locktime = locktime[::-1]

unsigned_tx = (
    version
    + input_count
    + inputs
    + output_count
    + outputs
    + locktime
)
print("unsigned_tx: ", unsigned_tx.hex())

# Decode the unsigned transaction to verify it looks correct
decoded = node.decoderawtransaction(unsigned_tx.hex())
print(json.dumps(decoded, indent=2, default=str))

Alice Delayed PubKey: d6363615b3d00361158c0f48a4ef81ea12e214e8d56e24098759f6b4267dca8b
Bob Revocation PubKey: 5460add4edf0961bfcc6288be0241114d7d90005db28753d835f2353f1529710
unsigned_tx:  030000000156bfa4a6be81aa71e797fba8db0117abba2e561de978b93906c5d82be3ccf67d0000000000fd33b48002f0000000000000000451024e7350410f0000000000225120e762c7f1d18ce3242936e755fdc4b5dc091d942387b647ad027bece2c294f24e6fa64320
{
  "txid": "b5f638bc5bf910d2fd2751a349554f9673938350383d8adb7df0f5aa05d23c2a",
  "hash": "b5f638bc5bf910d2fd2751a349554f9673938350383d8adb7df0f5aa05d23c2a",
  "version": 3,
  "size": 107,
  "vsize": 107,
  "weight": 428,
  "locktime": 541304431,
  "vin": [
    {
      "txid": "7df6cce32bd8c50639b978e91d562ebaab1701dba8fb97e771aa81bea6a4bf56",
      "vout": 0,
      "scriptSig": {
        "asm": "",
        "hex": ""
      },
      "sequence": 2159293437
    }
  ],
  "vout": [
    {
      "value": "0.00000240",
      "n": 0,
      "scriptPubKey": {
        "asm": "1 29518",
        "desc":

## The sighash for the key path spend

This is the “Schnorr key spend” case: you’re proving knowledge of the (tweaked) internal private key, with no script branch revealed. The message preimage is called msg_digest in [BIP-341](https://github.com/bitcoin/bips/blob/master/bip-0341.mediawiki).

In [8]:
index_of_this_input = bytes.fromhex("0000 0000")

# SIGHASH for key path spend
# See BIP-341 for details
sighash_epoch = bytes.fromhex("00")

# Control
hash_type = bytes.fromhex("00") # SIGHASH_DEFAULT (a new sighash type meaning implied SIGHASH_ALL)

# Transaction data
sha_prevouts = sha256(txid + index).digest()

sha_amounts = sha256(channel_value).digest()

# scriptPubKey P2TR: OP_1 (0x51) + PUSH32 (0x20) + xonly(32B)
sha_scriptpubkeys = sha256(
    varint_len(channel_spk)
    + channel_spk
).digest()

sha_sequences = sha256(sequence).digest()

sha_outputs = sha256(outputs).digest()

# Data about this input
spend_type = bytes.fromhex("00") # no annex present

sig_msg = (
    sighash_epoch
    + hash_type
    + version
    + locktime
    + sha_prevouts
    + sha_amounts
    + sha_scriptpubkeys
    + sha_sequences
    + sha_outputs
    + spend_type
    + index_of_this_input
)

tag_hash = sha256("TapSighash".encode()).digest()
sighash = sha256(tag_hash + tag_hash + sig_msg).digest()


## Signing the sighash

In [9]:
# Build participants to sort them using the same rule used in pubkeys aggregation.
participants = [
    (pubkey_alice_musig2.get_bytes(bip340=False), privkey_alice_musig2.get_bytes(), alice_per_commitment_seed),
    (pubkey_bob_musig2.get_bytes(bip340=False),   privkey_bob_musig2.get_bytes(), bob_per_commitment_seed),
]

# Reorder participants to match the sorted pubkeys
pk_to_tuple = {pk: (pk, sk, pcs) for pk, sk, pcs in participants}
participants = [pk_to_tuple[pk] for pk in sorted_pubkeys]

# Alice and Bob generates its own nonce pair (secnonce, pubnonce)
secnonces, pubnonces = [], []
for pk, sk, pcs in participants:
    # Use per-commitment nonce for Alice to get deterministic results
    if pk == pubkey_alice_musig2.get_bytes(bip340=True):
        sec, pub = nonce_per_commitment(pcs, commitment_number, sk, pk, agg_pubkey_tweaked, sighash)
    else:
        sec, pub = nonce_gen(sk, pk, agg_pubkey_tweaked, sighash, None)
    secnonces.append(sec)
    pubnonces.append(pub)

# Alice and Bob has exchanged already the pubnonces, so they can agregate them and create the session context
agg_nonce = nonce_agg(pubnonces)
session = SessionContext(agg_nonce, sorted_pubkeys, [bip86_tweak], [True], sighash)

# Alice and Bob produces their partial signatures
psigs = [sign(sec, sk, session) for sec, (_, sk, _) in zip(secnonces, participants)]

# Each side verify the other’s partial signature before proceeding
for i, psig in enumerate(psigs):
    assert partial_sig_verify(psig, pubnonces, sorted_pubkeys, [bip86_tweak], [True], sighash, i)

# Each side combines partial signatures into the final Schnorr signature
agg_sig = partial_sig_agg(psigs, session)

# Sanity check: verify with BIP340 against the *tweaked* x-only key ---
ok = schnorr_verify(sighash, agg_pubkey_tweaked, agg_sig)
print("aggregated Schnorr verifies?", ok)


aggregated Schnorr verifies? True


## The Commitment Signed Transaction

In [10]:
witness = (
    bytes.fromhex("01") # one stack item in the witness
    + varint_len(agg_sig)
    + agg_sig
)

# the final signed transaction
signed_tx = (
    version
    + marker
    + flag
    + input_count
    + inputs
    + output_count
    + outputs
    + witness
    + locktime
)

print("signed tx: ",signed_tx.hex())
# Decode the unsigned transaction to verify it looks correct
decoded = node.decoderawtransaction(signed_tx.hex())
print(json.dumps(decoded, indent=2, default=str))

print(node.testmempoolaccept(rawtxs=[signed_tx.hex()]))

signed tx:  0300000000010156bfa4a6be81aa71e797fba8db0117abba2e561de978b93906c5d82be3ccf67d0000000000fd33b48002f0000000000000000451024e7350410f0000000000225120e762c7f1d18ce3242936e755fdc4b5dc091d942387b647ad027bece2c294f24e01407841b4ec9205a273cfa039585fa2eb6e3c2d780663659434088ebb1216df7da8df4fa661fd68ab5398a84f3ac125b78bd1a6aa3b17e60f84da3adfe9ca0b45fe6fa64320
{
  "txid": "b5f638bc5bf910d2fd2751a349554f9673938350383d8adb7df0f5aa05d23c2a",
  "hash": "e46f3e2c20979b05e4ff4bd954594000da03909af69aad289f2b803f578580bf",
  "version": 3,
  "size": 175,
  "vsize": 124,
  "weight": 496,
  "locktime": 541304431,
  "vin": [
    {
      "txid": "7df6cce32bd8c50639b978e91d562ebaab1701dba8fb97e771aa81bea6a4bf56",
      "vout": 0,
      "scriptSig": {
        "asm": "",
        "hex": ""
      },
      "txinwitness": [
        "7841b4ec9205a273cfa039585fa2eb6e3c2d780663659434088ebb1216df7da8df4fa661fd68ab5398a84f3ac125b78bd1a6aa3b17e60f84da3adfe9ca0b45fe"
      ],
      "sequence": 2159293437
    }
 